# Notebook 03: Anomaly Detection on Real Data
## Retrain IF + OC-SVM + AE, Ensemble, Cross-Donor, Dose-Response
**Evaluation criteria:** ROC-AUC ≥ 0.95, Sensitivity ≥ 90%, Specificity ≥ 95%, LOD ≤ 10 CFU/mL

In [ ]:
import numpy as np, pandas as pd, struct
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, classification_report
from sklearn.model_selection import cross_val_score
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import warnings; warnings.filterwarnings('ignore')

BASE = Path('/run/media/sham/AI_/ai-stack/projects/biopharma-contamination-detection')
OUT = BASE / 'data' / 'processed'
FIG = BASE / 'figures'
FIG.mkdir(exist_ok=True)
df = pd.read_parquet(OUT / 'real_dataset.parquet')
print(f'Loaded {len(df)} spectra')

## Unpack spectra

In [ ]:
def unpack(row):    n = row['n_wl']    data = struct.unpack(f'{n*2}d', row['spectrum_bytes'])    return np.array(data[::2]), np.array(data[1::2])spectra_data = []for _, row in df.iterrows():    wl, ab = unpack(row)    spectra_data.append({'wavelengths': wl, 'absorbance': ab, 'label': row['label'],                         'organism': row['organism'], 'cfu': row['cfu'], 'donor_id': row['donor_id']})df_s = pd.DataFrame(spectra_data)# Use absorbance at each wavelength as features (interpolate to common grid)target_wl = np.arange(230, 610, 1)  # 230-609 nm, 1nm stepsdef interpolate_spectrum(row):    return np.interp(target_wl, row['wavelengths'], row['absorbance'])X = np.vstack(df_s['absorbance'].apply(interpolate_spectrum).values)y = df_s['label'].valuesprint(f'Feature matrix: {X.shape}')print(f'Labels: sterile={(y==0).sum}, contaminated={(y==1).sum}')

## 1. Isolation Forest

In [ ]:
X_clean = X[y == 0]
X_contam = X[y == 1]
print(f'Training on {len(X_clean)} clean, testing on {len(X_contam)} contaminated')

scaler = RobustScaler()
X_clean_s = scaler.fit_transform(X_clean)
X_all_s = scaler.transform(X)

iso = IsolationForest(n_estimators=200, contamination=0.01, random_state=42, n_jobs=-1)
iso.fit(X_clean_s)
scores_iso = -iso.score_samples(X_all_s)  # Higher = more anomalous
scores_iso_contam = scores_iso[y == 1]
scores_iso_clean = scores_iso[y == 0]

thresh = np.percentile(scores_iso_clean, 95)
preds_iso = (scores_iso > thresh).astype(int)
sens = (preds_iso[y==1] == 1).mean()
spec = (preds_iso[y==0] == 0).mean()
print(f'Isolation Forest: Sensitivity={sens:.1%}, Specificity={spec:.1%}')

## 2. One-Class SVM

In [ ]:
ocsvm = OneClassSVM(kernel='rbf', gamma=0.001, nu=0.05)
ocsvm.fit(X_clean_s)
scores_ocsvm = -ocsvm.decision_function(X_all_s)  # Higher = more anomalous
thresh_oc = np.percentile(scores_ocsvm[y==0], 95)
preds_oc = (scores_ocsvm > thresh_oc).astype(int)
sens_oc = (preds_oc[y==1] == 1).mean()
spec_oc = (preds_oc[y==0] == 0).mean()
print(f'One-Class SVM: Sensitivity={sens_oc:.1%}, Specificity={spec_oc:.1%}')

## 3. Deep Autoencoder

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, 16)
        )
        self.decoder = nn.Sequential(
            nn.Linear(16, 64), nn.ReLU(),
            nn.Linear(64, 128), nn.ReLU(),
            nn.Linear(128, input_dim)
        )
    def forward(self, x): return self.decoder(self.encoder(x))

torch.manual_seed(42)
ae = Autoencoder(X.shape[1])
opt = torch.optim.Adam(ae.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()
X_tensor = torch.FloatTensor(X_clean_s)
dl = DataLoader(TensorDataset(X_tensor), batch_size=32, shuffle=True)

for epoch in range(50):
    ae.train()
    total = 0
    for (batch,) in dl:
        opt.zero_grad()
        recon = ae(batch)
        loss = loss_fn(recon, batch)
        loss.backward(); opt.step()
        total += loss.item()
    if epoch % 10 == 0: print(f'  Epoch {epoch}: loss={total/len(dl):.6f}')

# Compute reconstruction errors
ae.eval()
with torch.no_grad():
    X_t = torch.FloatTensor(X_all_s)
    recon = ae(X_t)
    errors = torch.mean((X_t - recon)**2, dim=1).numpy()

thresh_ae = np.percentile(errors[y==0], 95)
preds_ae = (errors > thresh_ae).astype(int)
sens_ae = (preds_ae[y==1] == 1).mean()
spec_ae = (preds_ae[y==0] == 0).mean()
print(f'Autoencoder: Sensitivity={sens_ae:.1%}, Specificity={spec_ae:.1%}')

## 4. Weighted Ensemble

In [ ]:
from scipy.stats import rankdata

def norm_scores(s): return (s - s.min()) / (s.max() - s.min() + 1e-10)

s_iso = norm_scores(scores_iso)
s_oc = norm_scores(scores_ocsvm)
s_ae = norm_scores(errors)

# Learn weights via grid search on clean vs contaminated
best_auc, best_w = 0, (0.33, 0.33, 0.34)
for w1 in np.arange(0.1, 0.6, 0.1):
    for w2 in np.arange(0.1, 0.6, 0.1):
        w3 = 1 - w1 - w2
        if w3 < 0.05: continue
        ens = w1*s_iso + w2*s_oc + w3*s_ae
        auc = roc_auc_score(y, ens)
        if auc > best_auc:
            best_auc, best_w = auc, (w1, w2, w3)

w1, w2, w3 = best_w
ens_score = w1*s_iso + w2*s_oc + w3*s_ae
thresh_ens = np.percentile(ens_score[y==0], 95)
preds_ens = (ens_score > thresh_ens).astype(int)
sens_ens = (preds_ens[y==1] == 1).mean()
spec_ens = (preds_ens[y==0] == 0).mean()
auc_ens = roc_auc_score(y, ens_score)
print(f'Ensemble (w={w1:.2f},{w2:.2f},{w3:.2f}): ROC-AUC={auc_ens:.4f}, Sens={sens_ens:.1%}, Spec={spec_ens:.1%}')

## 5. ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
for scores, label, color in [(scores_iso,'Isolation Forest','#2196F3'),(scores_ocsvm,'One-Class SVM','#FF9800'),
                               (errors,'Autoencoder','#4CAF50'),(ens_score,'Ensemble','#E91E63')]:
    fpr, tpr, _ = roc_curve(y, scores)
    auc = roc_auc_score(y, scores)
    ax.plot(fpr, tpr, label=f'{label} (AUC={auc:.4f})', color=color, linewidth=2)
ax.plot([0,1],[0,1],'k--',alpha=0.3)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title(f'ROC Curves — Real UV-Vis Data (n={len(y)})', fontsize=14)
ax.legend(fontsize=11, loc='lower right')
plt.tight_layout()
fig.savefig(FIG / 'roc_curves_real.png', dpi=300)
print('Saved: roc_curves_real.png')
plt.close()

## 6. Cross-Donor Generalization

In [ ]:
donors = df['donor_id'].unique()
print(f'\nCross-donor test (leave-one-donor-out):')
donor_aucs = []
for donor in donors[:10]:  # Top 10 donors
    mask_test = df['donor_id'] == donor
    mask_train = ~mask_test
    if mask_test.sum() < 5 or mask_train.sum() < 50: continue
    X_tr = X[mask_train.values]; y_tr = y[mask_train.values]
    X_te = X[mask_test.values]; y_te = y[mask_test.values]
    sc_tr = RobustScaler().fit(X_tr[y_tr==0])
    iso2 = IsolationForest(n_estimators=100, contamination=0.01, random_state=42)
    iso2.fit(sc_tr.transform(X_tr[y_tr==0]))
    scores = -iso2.score_samples(sc_tr.transform(X_te))
    if y_te.sum() > 0:
        auc = roc_auc_score(y_te, scores)
        donor_aucs.append((donor, auc, len(y_te)))
        print(f'  {donor}: AUC={auc:.4f} (n={len(y_te)})')
if donor_aucs:
    print(f'\nMean cross-donor AUC: {np.mean([a for _,a,_ in donor_aucs]):.4f}')

## 7. Dose-Response Curve

In [ ]:
cfu_levels = sorted(df_s[df_s['cfu']>0]['cfu'].unique())
print(f'\nDose-response by CFU level:')
det_rates = []
for cfu in cfu_levels:
    mask = df_s['cfu'] == cfu
    if mask.sum() == 0: continue
    s = ens_score[mask.values]
    det_rate = (s > thresh_ens).mean()
    det_rates.append({'cfu': cfu, 'detection_rate': det_rate, 'n': mask.sum()})
    print(f'  {cfu} CFU/mL: detection rate = {det_rate:.1%} (n={mask.sum()})')

df_dr = pd.DataFrame(det_rates)
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(df_dr['cfu'], df_dr['detection_rate'], 'o-', linewidth=2, markersize=10)
ax.axhline(0.9, color='red', linestyle='--', alpha=0.5, label='90% detection threshold')
ax.set_xlabel('CFU/mL', fontsize=12); ax.set_ylabel('Detection Rate', fontsize=12)
ax.set_title('Dose-Response: Contamination Detection Rate vs CFU', fontsize=14)
ax.legend(); ax.set_xscale('log')
plt.tight_layout()
fig.savefig(FIG / 'dose_response.png', dpi=300)
print('Saved: dose_response.png')
plt.close()

# LOD90
lod90 = None
for _, r in df_dr.iterrows():
    if r['detection_rate'] >= 0.9:
        lod90 = r['cfu']; break
print(f'\nLOD90 (90% detection rate): {lod90} CFU/mL' if lod90 else '\nLOD90: Not reached')

## Evaluation Summary

In [ ]:
print('\n' + '='*60)
print('EVALUATION CRITERIA')
print('='*60)
print(f'ROC-AUC:        {auc_ens:.4f}  (target: ≥ 0.95)  {"✅ PASS" if auc_ens >= 0.95 else "❌ FAIL"}')
print(f'Sensitivity:    {sens_ens:.1%}  (target: ≥ 90%)   {"✅ PASS" if sens_ens >= 0.90 else "❌ FAIL"}')
print(f'Specificity:    {spec_ens:.1%}  (target: ≥ 95%)   {"✅ PASS" if spec_ens >= 0.95 else "❌ FAIL"}')
print(f'LOD90:          {lod90} CFU/mL  (target: ≤ 10)   {"✅ PASS" if lod90 and lod90 <= 10 else "⚠️  NEEDS WORK"}')
print('='*60)